In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

DATA_DIR = "/content/drive/MyDrive/zepto-data-ai-platform/data_pipeline"

os.makedirs(DATA_DIR, exist_ok=True)

print("Module 1 folder:", DATA_DIR)

Module 1 folder: /content/drive/MyDrive/zepto-data-ai-platform/data_pipeline


In [4]:
!pip install -q requests beautifulsoup4 pandas
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time

In [5]:
BASE_URL = "https://books.toscrape.com/"
GBP_TO_INR = 105.50

print("GBP to INR rate:", GBP_TO_INR)

GBP to INR rate: 105.5


In [6]:
TARGET_CATEGORIES = [
    "Fiction",
    "Mystery",
    "Historical Fiction"
]

In [7]:
def get_soup(url):
    response = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    response.raise_for_status()

    return BeautifulSoup(response.text, "html.parser")

In [8]:
soup = get_soup(BASE_URL)

category_urls = {}

for link in soup.select(".side_categories ul li ul li a"):
    category_name = link.get_text(strip=True)

    if category_name in TARGET_CATEGORIES:
        category_urls[category_name] = BASE_URL + link["href"]

category_urls

{'Mystery': 'https://books.toscrape.com/catalogue/category/books/mystery_3/index.html',
 'Historical Fiction': 'https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html',
 'Fiction': 'https://books.toscrape.com/catalogue/category/books/fiction_10/index.html'}

In [9]:
def scrape_category(category_name, category_url):

    books = []
    current_url = category_url

    while current_url:

        print("Scraping:", category_name, current_url)

        soup = get_soup(current_url)

        book_cards = soup.select("article.product_pod")

        for book in book_cards:

            title = book.h3.a["title"].strip()

            price = (
                book.select_one(".price_color")
                .get_text(strip=True)
            )

            rating_classes = book.select_one(".star-rating").get("class", [])

            star_rating = None

            for value in rating_classes:
                if value != "star-rating":
                    star_rating = value

            availability = (
                book.select_one(".availability")
                .get_text(" ", strip=True)
            )

            books.append({
                "title": title,
                "price": price,
                "star_rating": star_rating,
                "availability": availability,
                "category": category_name
            })

        next_button = soup.select_one("li.next a")

        if next_button:
            base_folder = current_url.rsplit("/", 1)[0]
            current_url = base_folder + "/" + next_button["href"]
        else:
            current_url = None

        time.sleep(0.2)

    return books

In [10]:
all_books = []

for category in TARGET_CATEGORIES:

    books = scrape_category(
        category,
        category_urls[category]
    )

    all_books.extend(books)

df = pd.DataFrame(all_books)

print("Total books:", len(df))
print()
print(df["category"].value_counts())

df.head()

Scraping: Fiction https://books.toscrape.com/catalogue/category/books/fiction_10/index.html
Scraping: Fiction https://books.toscrape.com/catalogue/category/books/fiction_10/page-2.html
Scraping: Fiction https://books.toscrape.com/catalogue/category/books/fiction_10/page-3.html
Scraping: Fiction https://books.toscrape.com/catalogue/category/books/fiction_10/page-4.html
Scraping: Mystery https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Scraping: Mystery https://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html
Scraping: Historical Fiction https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Scraping: Historical Fiction https://books.toscrape.com/catalogue/category/books/historical-fiction_4/page-2.html
Total books: 123

category
Fiction               65
Mystery               32
Historical Fiction    26
Name: count, dtype: int64


,title,price,star_rating,availability,category
0,Soumission,Â£50.10,One,In stock,Fiction
1,Private Paris (Private #10),Â£47.61,Five,In stock,Fiction
2,"We Love You, Charlie Freeman",Â£50.27,Five,In stock,Fiction
3,Thirst,Â£17.27,Five,In stock,Fiction
4,The Murder That Never Was (Forensic Instincts #5),Â£54.11,Three,In stock,Fiction


In [11]:
# Create a copy so our original scraped data remains unchanged
df_clean = df.copy()

# --------------------------------
# 1. Clean price
# --------------------------------
# Extract only the numeric part.
# This also handles values displayed like Â£50.10
df_clean["price_gbp"] = (
    df_clean["price"]
    .astype(str)
    .str.extract(r"(\d+\.\d+)")[0]
)

df_clean["price_gbp"] = pd.to_numeric(
    df_clean["price_gbp"],
    errors="coerce"
)

# --------------------------------
# 2. Convert star rating to integer
# --------------------------------
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df_clean["rating"] = df_clean["star_rating"].map(rating_map)

# --------------------------------
# 3. Convert availability to boolean
# --------------------------------
df_clean["in_stock"] = (
    df_clean["availability"]
    .astype(str)
    .str.lower()
    .str.contains("in stock")
)

# --------------------------------
# 4. Handle parsing failures
# --------------------------------
print("Missing price values before cleaning:",
      df_clean["price_gbp"].isna().sum())

print("Missing rating values before cleaning:",
      df_clean["rating"].isna().sum())

# Numeric parsing failures -> median imputation
if df_clean["price_gbp"].isna().any():
    median_price = df_clean["price_gbp"].median()
    df_clean["price_gbp"] = df_clean["price_gbp"].fillna(median_price)

if df_clean["rating"].isna().any():
    median_rating = df_clean["rating"].median()
    df_clean["rating"] = df_clean["rating"].fillna(median_rating)

# Rating should be integer
df_clean["rating"] = df_clean["rating"].round().astype(int)

# --------------------------------
# 5. GBP -> INR
# Required project conversion:
# 1 GBP = 105.50 INR
# --------------------------------
GBP_TO_INR = 105.50

df_clean["price_inr"] = (
    df_clean["price_gbp"] * GBP_TO_INR
).round(2)

# --------------------------------
# 6. Keep final required columns
# --------------------------------
df_clean = df_clean[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category"
    ]
]

df_clean.head()

Missing price values before cleaning: 0
Missing rating values before cleaning: 0


,title,price_gbp,price_inr,rating,in_stock,category
0,Soumission,50.10,5285.55,1,True,Fiction
1,Private Paris (Private #10),47.61,5022.85,5,True,Fiction
2,"We Love You, Charlie Freeman",50.27,5303.48,5,True,Fiction
3,Thirst,17.27,1821.98,5,True,Fiction
4,The Murder That Never Was (Forensic Instincts #5),54.11,5708.60,3,True,Fiction


In [12]:
print("DATA TYPES")
print(df_clean.dtypes)

print("\nMISSING VALUES")
print(df_clean.isnull().sum())

print("\nNUMBER OF BOOKS:", len(df_clean))
print("NUMBER OF CATEGORIES:", df_clean["category"].nunique())

print("\nBOOKS PER CATEGORY")
print(df_clean["category"].value_counts())

DATA TYPES
title         object
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      object
dtype: object

MISSING VALUES
title        0
price_gbp    0
price_inr    0
rating       0
in_stock     0
category     0
dtype: int64

NUMBER OF BOOKS: 123
NUMBER OF CATEGORIES: 3

BOOKS PER CATEGORY
category
Fiction               65
Mystery               32
Historical Fiction    26
Name: count, dtype: int64


In [14]:
assert len(df_clean) >= 60, "Need at least 60 books"

assert df_clean["category"].nunique() >= 3, \
    "Need at least 3 categories"

assert df_clean["rating"].between(1, 5).all(), \
    "Ratings must be between 1 and 5"

assert df_clean["price_gbp"].notna().all(), \
    "price_gbp contains missing values"

assert df_clean["price_inr"].notna().all(), \
    "price_inr contains missing values"

print("At least 60 books")
print("At least 3 categories")
print("Ratings are between 1 and 5")
print("Prices cleaned successfully")
print("GBP to INR conversion completed")
print("\nCleaning validation PASSED.")

At least 60 books
At least 3 categories
Ratings are between 1 and 5
Prices cleaned successfully
GBP to INR conversion completed

Cleaning validation PASSED.


In [15]:
CSV_PATH = f"{DATA_DIR}/books_cleaned.csv"

df_clean.to_csv(
    CSV_PATH,
    index=False
)

print("CSV saved successfully!")
print(CSV_PATH)

CSV saved successfully!
/content/drive/MyDrive/zepto-data-ai-platform/data_pipeline/books_cleaned.csv


In [23]:
import sqlite3
import os

DB_PATH = f"{DATA_DIR}/books.db"

# Remove old database if rerunning the notebook
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)

cursor = conn.cursor()

# Enable foreign keys
cursor.execute("PRAGMA foreign_keys = ON")

# -----------------------------
# Categories table
# -----------------------------
cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

# -----------------------------
# Books table
# -----------------------------
cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,

    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

conn.commit()

print("Database and tables created.")

Database and tables created.


In [24]:
# Insert unique categories
categories = sorted(df_clean["category"].unique())

for category in categories:
    cursor.execute(
        """
        INSERT INTO categories (category_name)
        VALUES (?)
        """,
        (category,)
    )

conn.commit()


# Get category IDs
cursor.execute("""
SELECT category_id, category_name
FROM categories
""")

category_map = {
    name: category_id
    for category_id, name in cursor.fetchall()
}

print("Category Mapping:")
print(category_map)


# Insert books
for _, row in df_clean.iterrows():

    cursor.execute(
        """
        INSERT INTO books (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            row["title"],
            float(row["price_gbp"]),
            float(row["price_inr"]),
            int(row["rating"]),
            int(row["in_stock"]),
            category_map[row["category"]]
        )
    )

conn.commit()

print("\nBooks inserted successfully.")

Category Mapping:
{'Fiction': 1, 'Historical Fiction': 2, 'Mystery': 3}

Books inserted successfully.


In [25]:
book_count = cursor.execute(
    "SELECT COUNT(*) FROM books"
).fetchone()[0]

category_count = cursor.execute(
    "SELECT COUNT(*) FROM categories"
).fetchone()[0]

print("Books in database:", book_count)
print("Categories in database:", category_count)

Books in database: 123
Categories in database: 3


In [26]:
queries = {

    "Query 1 - Five Star Books": """
        SELECT title, rating, price_gbp
        FROM books
        WHERE rating = 5
        LIMIT 10;
    """,

    "Query 2 - Most Expensive Books": """
        SELECT title, price_gbp, price_inr
        FROM books
        ORDER BY price_gbp DESC
        LIMIT 10;
    """,

    "Query 3 - Distinct Categories": """
        SELECT DISTINCT category_name
        FROM categories;
    """,

    "Query 4 - Books Between GBP 20 and 40": """
        SELECT title, price_gbp, rating
        FROM books
        WHERE price_gbp BETWEEN 20 AND 40
        ORDER BY price_gbp ASC
        LIMIT 15;
    """,

    "Query 5 - Rating 4 or 5": """
        SELECT title, rating, price_gbp
        FROM books
        WHERE rating IN (4, 5)
        ORDER BY rating DESC
        LIMIT 15;
    """,

    "Query 6 - JOIN Books and Categories": """
        SELECT
            b.book_id,
            b.title,
            b.price_gbp,
            b.price_inr,
            b.rating,
            b.in_stock,
            c.category_name
        FROM books b
        INNER JOIN categories c
            ON b.category_id = c.category_id
        ORDER BY b.book_id;
    """
}


for query_name, query in queries.items():

    print("\n" + "=" * 60)
    print(query_name)
    print("=" * 60)

    result = pd.read_sql_query(
        query,
        conn
    )

    display(result)


Query 1 - Five Star Books


,title,rating,price_gbp
0,Private Paris (Private #10),5,47.61
1,"We Love You, Charlie Freeman",5,50.27
2,Thirst,5,17.27
3,The Regional Office Is Under Attack!,5,51.36
4,Finders Keepers (Bill Hodges Trilogy #2),5,53.53
5,The Time Keeper,5,27.88
6,Dear Mr. Knightley,5,11.21
7,The Silent Sister (Riley MacPherson #1),5,46.29
8,Siddhartha,5,34.22
9,Digital Fortress,5,58.00



Query 2 - Most Expensive Books


,title,price_gbp,price_inr
0,Last One Home (New Beginnings #1),59.98,6327.89
1,Boar Island (Anna Pigeon #19),59.48,6275.14
2,The Improbability of Love,59.45,6271.98
3,Miller's Valley,58.54,6175.97
4,Digital Fortress,58.00,6119.00
5,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70,6087.35
6,Kitchens of the Great Midwest,57.20,6034.60
7,The Dinner Party,56.54,5964.97
8,The Past Never Ends,56.50,5960.75
9,Shtum,55.84,5891.12



Query 3 - Distinct Categories


...

,title,price_gbp,rating
0,Blood Defense (Samantha Brinkman #1),20.30,3
1,"Love, Lies and Spies",20.55,2
2,Between Shades of Gray,20.79,5
3,Delivering the Truth (Quaker Midwife Mystery #1),20.89,4
4,Tuesday Nights in 1980,21.04,2
5,Voyager (Outlander #3),21.07,5
6,Hystopia: A Novel,21.96,4
7,The Art of Fielding,22.10,1
8,Big Little Lies,22.11,1
9,The Da Vinci Code (Robert Langdon #2),22.96,2



Query 5 - Rating 4 or 5


,title,rating,price_gbp
0,Private Paris (Private #10),5,47.61
1,"We Love You, Charlie Freeman",5,50.27
2,Thirst,5,17.27
3,The Regional Office Is Under Attack!,5,51.36
4,Finders Keepers (Bill Hodges Trilogy #2),5,53.53
5,The Time Keeper,5,27.88
6,Dear Mr. Knightley,5,11.21
7,The Silent Sister (Riley MacPherson #1),5,46.29
8,Siddhartha,5,34.22
9,Digital Fortress,5,58.00



Query 6 - JOIN Books and Categories


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,1,Soumission,50.10,5285.55,1,1,Fiction
1,2,Private Paris (Private #10),47.61,5022.85,5,1,Fiction
2,3,"We Love You, Charlie Freeman",50.27,5303.48,5,1,Fiction
3,4,Thirst,17.27,1821.98,5,1,Fiction
4,5,The Murder That Never Was (Forensic Instincts #5),54.11,5708.60,3,1,Fiction
...,...,...,...,...,...,...,...
118,119,While You Were Mine,41.32,4359.26,5,1,Historical Fiction
119,120,The Secret Healer,34.56,3646.08,3,1,Historical Fiction
120,121,Starlark,25.83,2725.06,3,1,Historical Fiction
121,122,Lost Among the Living,27.70,2922.35,4,1,Historical Fiction


In [27]:
QUERY_OUTPUT_PATH = f"{DATA_DIR}/query_outputs.txt"

with open(
    QUERY_OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as file:

    for query_name, query in queries.items():

        result = pd.read_sql_query(
            query,
            conn
        )

        file.write("\n")
        file.write("=" * 70 + "\n")
        file.write(query_name + "\n")
        file.write("=" * 70 + "\n")

        file.write("\nSQL QUERY:\n")
        file.write(query.strip())

        file.write("\n\nOUTPUT:\n")
        file.write(
            result.to_string(index=False)
        )

        file.write("\n\n")


print("✅ SQL queries and outputs saved.")
print(QUERY_OUTPUT_PATH)

✅ SQL queries and outputs saved.
/content/drive/MyDrive/zepto-data-ai-platform/data_pipeline/query_outputs.txt


In [28]:
# SQL result 1
five_star_df = pd.read_sql("""
SELECT title, rating, price_gbp
FROM books
WHERE rating = 5
LIMIT 10
""", conn)

print("QUERY 1 USING pd.read_sql()")
display(five_star_df)


# SQL result 2
expensive_df = pd.read_sql("""
SELECT title, price_gbp
FROM books
ORDER BY price_gbp DESC
LIMIT 10
""", conn)

print("QUERY 2 USING pd.read_sql()")
display(expensive_df)

QUERY 1 USING pd.read_sql()


,title,rating,price_gbp
0,Private Paris (Private #10),5,47.61
1,"We Love You, Charlie Freeman",5,50.27
2,Thirst,5,17.27
3,The Regional Office Is Under Attack!,5,51.36
4,Finders Keepers (Bill Hodges Trilogy #2),5,53.53
5,The Time Keeper,5,27.88
6,Dear Mr. Knightley,5,11.21
7,The Silent Sister (Riley MacPherson #1),5,46.29
8,Siddhartha,5,34.22
9,Digital Fortress,5,58.00


QUERY 2 USING pd.read_sql()


,title,price_gbp
0,Last One Home (New Beginnings #1),59.98
1,Boar Island (Anna Pigeon #19),59.48
2,The Improbability of Love,59.45
3,Miller's Valley,58.54
4,Digital Fortress,58.00
5,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70
6,Kitchens of the Great Midwest,57.20
7,The Dinner Party,56.54
8,The Past Never Ends,56.50
9,Shtum,55.84


In [29]:
# ---------------------------------
# SQL JOIN
# ---------------------------------

sql_join_df = pd.read_sql("""
SELECT
    b.book_id,
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books b
INNER JOIN categories c
    ON b.category_id = c.category_id
ORDER BY b.book_id
""", conn)


# ---------------------------------
# Read individual tables
# ---------------------------------

books_df = pd.read_sql(
    "SELECT * FROM books",
    conn
)

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)


# ---------------------------------
# Reproduce JOIN with pd.merge()
# ---------------------------------

pandas_join_df = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)


pandas_join_df = pandas_join_df[
    [
        "book_id",
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_name"
    ]
]

pandas_join_df = (
    pandas_join_df
    .sort_values("book_id")
    .reset_index(drop=True)
)

sql_join_df = (
    sql_join_df
    .reset_index(drop=True)
)


print("SQL JOIN RESULT")
display(sql_join_df.head(10))


print("\nPANDAS pd.merge() RESULT")
display(pandas_join_df.head(10))

SQL JOIN RESULT


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,1,Soumission,50.10,5285.55,1,1,Fiction
1,2,Private Paris (Private #10),47.61,5022.85,5,1,Fiction
2,3,"We Love You, Charlie Freeman",50.27,5303.48,5,1,Fiction
3,4,Thirst,17.27,1821.98,5,1,Fiction
4,5,The Murder That Never Was (Forensic Instincts #5),54.11,5708.60,3,1,Fiction
5,6,Tuesday Nights in 1980,21.04,2219.72,2,1,Fiction
6,7,The Vacationers,42.15,4446.82,4,1,Fiction
7,8,The Regional Office Is Under Attack!,51.36,5418.48,5,1,Fiction
8,9,Finders Keepers (Bill Hodges Trilogy #2),53.53,5647.42,5,1,Fiction
9,10,The Time Keeper,27.88,2941.34,5,1,Fiction



PANDAS pd.merge() RESULT


,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,1,Soumission,50.10,5285.55,1,1,Fiction
1,2,Private Paris (Private #10),47.61,5022.85,5,1,Fiction
2,3,"We Love You, Charlie Freeman",50.27,5303.48,5,1,Fiction
3,4,Thirst,17.27,1821.98,5,1,Fiction
4,5,The Murder That Never Was (Forensic Instincts #5),54.11,5708.60,3,1,Fiction
5,6,Tuesday Nights in 1980,21.04,2219.72,2,1,Fiction
6,7,The Vacationers,42.15,4446.82,4,1,Fiction
7,8,The Regional Office Is Under Attack!,51.36,5418.48,5,1,Fiction
8,9,Finders Keepers (Bill Hodges Trilogy #2),53.53,5647.42,5,1,Fiction
9,10,The Time Keeper,27.88,2941.34,5,1,Fiction


In [30]:
match = sql_join_df.equals(
    pandas_join_df
)

print("Do SQL JOIN and pd.merge() results match?")

print(match)

if match:
    print("PASS - Both approaches produce equivalent output.")
else:
    print("Results differ.")

Do SQL JOIN and pd.merge() results match?
True
PASS - Both approaches produce equivalent output.


In [31]:
import os

print("DATA_DIR =", DATA_DIR)
print("\nFiles currently in data_pipeline:")
print(os.listdir(DATA_DIR))

print("\nDatabase path:")
print(DB_PATH)

print("\nDoes books.db exist?")
print(os.path.exists(DB_PATH))

DATA_DIR = /content/drive/MyDrive/zepto-data-ai-platform/data_pipeline

Files currently in data_pipeline:
['module1_data_pipeline.ipynb', 'books_cleaned.csv', 'books.db', 'query_outputs.txt']

Database path:
/content/drive/MyDrive/zepto-data-ai-platform/data_pipeline/books.db

Does books.db exist?
True


In [35]:
print("========== MODULE 1 FINAL CHECK ==========")

print("Total books:", len(df_clean))
print("Total categories:", df_clean["category"].nunique())
print("Columns:", list(df_clean.columns))

print("\nData Types:")
print(df_clean.dtypes)

print("\nMissing Values:")
print(df_clean.isnull().sum())

print("\nGBP to INR rate used:", GBP_TO_INR)

print("\nSQL JOIN = Pandas Merge:", match)

assert len(df_clean) >= 60
assert df_clean["category"].nunique() >= 3
assert match == True

print("\nMODULE 1 VALIDATION PASSED")

========== MODULE 1 FINAL CHECK ==========
Total books: 123
Total categories: 3
Columns: ['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category']

Data Types:
title         object
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      object
dtype: object

Missing Values:
title        0
price_gbp    0
price_inr    0
rating       0
in_stock     0
category     0
dtype: int64

GBP to INR rate used: 105.5

SQL JOIN = Pandas Merge: True

MODULE 1 VALIDATION PASSED


In [38]:
from pathlib import Path

README_PATH = Path(DATA_DIR) / "README.md"

readme_text = r'''# Module 1 — Data Pipeline

## Overview

This module implements an end-to-end data engineering pipeline using Books to Scrape.

The pipeline includes:
1. Web scraping
2. Data cleaning
3. Currency conversion
4. SQLite database creation
5. SQL analysis
6. Pandas SQL and merge validation

## Data Source

Source: https://books.toscrape.com/

Three categories were scraped:
- Fiction
- Mystery
- Historical Fiction

The final dataset contains 123 books across 3 categories.

## Fields Scraped

- title
- price
- star_rating
- availability
- category

## Data Cleaning

The scraped price was converted into a numeric `price_gbp` column.

Star ratings were converted as:
- One = 1
- Two = 2
- Three = 3
- Four = 4
- Five = 5

Availability was converted into the Boolean `in_stock` column.

If a numeric parsing failure occurs, median imputation is used to prevent
unexpected values from stopping the pipeline.

## Currency Conversion

The project-defined fixed conversion rate is:

**1 GBP = 105.50 INR**

Formula:

`price_inr = price_gbp * 105.50`

No live currency API is required.

## Database Design

The SQLite database contains two normalized tables.

### categories

- category_id — Primary Key
- category_name

### books

- book_id — Primary Key
- title
- price_gbp
- price_inr
- rating
- in_stock
- category_id — Foreign Key

The `category_id` field creates the relationship between books and categories.

## SQL Analysis

The project demonstrates:
- SELECT
- WHERE
- ORDER BY
- LIMIT
- DISTINCT
- BETWEEN
- IN
- INNER JOIN

Six SQL queries and their outputs are saved in `query_outputs.txt`.

## Pandas Validation

Two SQL query results are loaded using `pd.read_sql()`.

The SQL JOIN is also recreated using `pd.merge()`.

The two outputs were compared and confirmed to be equivalent.

Result:

`SQL JOIN == pd.merge(): True`

## Files

- module1_data_pipeline.ipynb
- books_cleaned.csv
- books.db
- query_outputs.txt
- README.md
- requirements.txt

## How to Run

Install the required packages:

`pip install requests beautifulsoup4 pandas`

Then run all cells in `module1_data_pipeline.ipynb` from top to bottom.
'''

README_PATH.write_text(readme_text, encoding="utf-8")

print("README.md created successfully")
print(README_PATH)

README.md created successfully
/content/drive/MyDrive/zepto-data-ai-platform/data_pipeline/README.md


In [39]:
import os
print(os.listdir(DATA_DIR))

['module1_data_pipeline.ipynb', 'books_cleaned.csv', 'books.db', 'query_outputs.txt', 'README.md']


In [41]:
requirements = """requests
beautifulsoup4
pandas
"""

with open(f"{DATA_DIR}/requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created")

requirements.txt created


In [42]:
import os

DB_PATH = f"{DATA_DIR}/books.db"

print("Database exists:", os.path.exists(DB_PATH))
print("Database path:", DB_PATH)

Database exists: True
Database path: /content/drive/MyDrive/zepto-data-ai-platform/data_pipeline/books.db


In [43]:
print(os.listdir(DATA_DIR))

['module1_data_pipeline.ipynb', 'books_cleaned.csv', 'books.db', 'query_outputs.txt', 'README.md', 'requirements.txt']
